# Phase 0.5 — Data Exploration

Understand the source data before building transformations: schema, distributions, nulls, duplicates, cardinality, and watermark coverage.

**⚠️ PII handling:** `patients.name` (direct identifier) and `patients.postcode` (quasi-identifier) are masked throughout — null rates are computed from the column but raw values are never displayed. See `docs-claude/data_contract.md` § PII Fields.

**Findings from this notebook gate Phase 3 and Phase 4 design.** Do not design `intm.*` or `serve.*` models until the findings summary cell is complete.

In [17]:
from __future__ import annotations

import os
import warnings

import pandas as pd
import psycopg

warnings.filterwarnings("ignore")

POSTGRES_DSN = (
    f"postgresql://{os.getenv('POSTGRES_USER', 'postgres')}"
    f":{os.getenv('POSTGRES_PASSWORD', 'dev')}"
    f"@{os.getenv('POSTGRES_HOST', 'localhost')}"
    f":{os.getenv('POSTGRES_PORT', '5432')}"
    f"/{os.getenv('POSTGRES_DB', 'postgres')}"
)

conn = psycopg.connect(POSTGRES_DSN)


def query(sql: str) -> pd.DataFrame:
    """Execute sql and return results as a DataFrame."""
    with conn.cursor() as cur:
        cur.execute(sql)
        cols = [desc[0] for desc in cur.description]
        rows = cur.fetchall()
    return pd.DataFrame(rows, columns=cols)


def null_rates(df: pd.DataFrame) -> pd.Series:
    """Return null percentage per column, rounded to 2 decimal places."""
    return (df.isnull().mean() * 100).round(2)


print(f"Connected: {POSTGRES_DSN.split('@')[1]}")

Connected: localhost:5432/postgres


---
## 1. `appointments`

**Key question:** Does `event_id` carry duplicates (at-least-once delivery)? What is the status lifecycle?

In [18]:
df_apt = query("SELECT * FROM appointments")

print("=== Schema ===")
print(df_apt.dtypes)
print(f"\nRow count: {len(df_apt):,}")

=== Schema ===
event_id                               str
appointment_id                         str
patient_id                             str
provider_id                            str
scheduled_at       datetime64[us, Etc/UTC]
status                                 str
event_timestamp    datetime64[us, Etc/UTC]
ingested_at        datetime64[us, Etc/UTC]
dtype: object

Row count: 841


In [19]:
print("=== Null rates (%) ===")
print(null_rates(df_apt).to_string())

=== Null rates (%) ===
event_id           0.0
appointment_id     0.0
patient_id         0.0
provider_id        0.0
scheduled_at       0.0
status             0.0
event_timestamp    0.0
ingested_at        0.0


In [20]:
total = len(df_apt)
n_unique_event_id = df_apt["event_id"].nunique()
n_unique_apt_id = df_apt["appointment_id"].nunique()
n_dupes = total - n_unique_event_id

print("=== Duplicate analysis ===")
print(f"Total rows:              {total:,}")
print(f"Unique event_id:         {n_unique_event_id:,}")
print(f"Duplicate event_id rows: {n_dupes:,}  ({n_dupes / total * 100:.1f}%)")
print(f"Unique appointment_id:   {n_unique_apt_id:,}")
print(f"Avg events per appt:     {total / n_unique_apt_id:.2f}")

=== Duplicate analysis ===
Total rows:              841
Unique event_id:         812
Duplicate event_id rows: 29  (3.4%)
Unique appointment_id:   500
Avg events per appt:     1.68


In [21]:
# Investigate the nature of event_id duplicates.
#
# Two distinct patterns can produce non-unique event_id counts:
#   (A) Exact re-delivery  — same event_id, every column identical   → at-least-once duplicate
#   (B) Content collision  — same event_id, different status/timestamps → data integrity problem
#
# Separately, same appointment_id + DIFFERENT event_id = legitimate status-change event.

dup_ids = df_apt[df_apt.duplicated(subset=["event_id"], keep=False)]["event_id"].unique()
dups_df = df_apt[df_apt["event_id"].isin(dup_ids)].copy()
fully_identical = dups_df.duplicated(keep=False).all()

print("=== event_id duplicate investigation ===")
print(f"Distinct event_ids that appear >1 time: {len(dup_ids)}")
print(f"All duplicate rows are exact copies:    {fully_identical}")
print()

if fully_identical:
    print("Conclusion: duplicates are at-least-once re-deliveries (same payload emitted twice).")
    print("Pipeline dedup on event_id is safe — no information is lost by dropping the copy.")
else:
    print("⚠️  Some duplicates differ in content — investigate before deduplicating.")

# Show one example duplicate pair to confirm.
sample_id = dup_ids[0]
print(f"\n--- Sample duplicate (event_id={sample_id}) ---")
print(df_apt[df_apt["event_id"] == sample_id].to_string(index=False))

=== event_id duplicate investigation ===
Distinct event_ids that appear >1 time: 29
All duplicate rows are exact copies:    True

Conclusion: duplicates are at-least-once re-deliveries (same payload emitted twice).
Pipeline dedup on event_id is safe — no information is lost by dropping the copy.

--- Sample duplicate (event_id=d85d4040-fb0c-45a9-95dd-6e8c4e921971) ---
                            event_id appointment_id patient_id provider_id                     scheduled_at    status                  event_timestamp                      ingested_at
d85d4040-fb0c-45a9-95dd-6e8c4e921971        AP00021     PT0010       PR001 2026-04-09 06:11:21.167810+00:00 scheduled 2026-04-09 06:11:21.167810+00:00 2026-04-09 06:11:21.167810+00:00
d85d4040-fb0c-45a9-95dd-6e8c4e921971        AP00021     PT0010       PR001 2026-04-09 06:11:21.167810+00:00 scheduled 2026-04-09 06:11:21.167810+00:00 2026-04-09 06:11:21.167810+00:00


In [ ]:
# Confirm that status-change events (same appointment_id, different event_id) are NOT duplicates.
#
# Each appointment transitions through states (scheduled → completed/cancelled/no_show),
# and each transition is a separate event with its own unique event_id.

multi_event = (
    df_apt.groupby("appointment_id")["event_id"]
    .nunique()
    .rename("distinct_event_ids")
)
sample_apt_id = multi_event[multi_event > 1].index[0]

print("=== Status-change events (same appointment_id, distinct event_id) ===")
print(f"Example appointment_id: {sample_apt_id}")
print(
    df_apt[df_apt["appointment_id"] == sample_apt_id][
        ["event_id", "status", "event_timestamp", "ingested_at"]
    ].sort_values("event_timestamp").to_string(index=False)
)
print()
print("Each row has a distinct event_id — these are NOT duplicates.")
print("event_id uniqueness holds within each status-change event; dedup on event_id is correct.")

In [5]:
print("=== Status distribution ===")
print(df_apt["status"].value_counts().to_string())

print("\n=== Events per appointment_id (status-change history) ===")
events_per_apt = df_apt.groupby("appointment_id").size()
print(events_per_apt.value_counts().sort_index().rename("count of appointment_ids").to_string())

=== Status distribution ===
status
scheduled    511
completed    116
cancelled    107
no_show      107

=== Events per appointment_id (status-change history) ===
1    177
2    305
3     18


In [6]:
print("=== Timestamp coverage ===")
for col in ["scheduled_at", "event_timestamp", "ingested_at"]:
    series = df_apt[col]
    print(f"{col}:")
    print(f"  min={series.min()}  max={series.max()}  null={series.isnull().sum()}")

=== Timestamp coverage ===
scheduled_at:
  min=2026-02-08 06:11:21.167810+00:00  max=2026-05-09 06:11:21.167810+00:00  null=0
event_timestamp:
  min=2026-02-08 06:11:21.167810+00:00  max=2026-05-10 12:11:21.167810+00:00  null=0
ingested_at:
  min=2026-02-08 06:11:21.167810+00:00  max=2026-05-12 09:11:21.167810+00:00  null=0


---
## 2. `patients`

**Key question:** How many patients have multiple rows (SCD history)? Are `updated_at` watermarks populated?

**⚠️ PII:** `patients.name` is excluded from display. `patients.postcode` is shown as a null rate only — values are masked.

In [7]:
# Fetch all columns for statistics; name is never displayed
_df_pts_full = query("SELECT * FROM patients")

# Safe projection — drop name before any display
df_pts = _df_pts_full.drop(columns=["name"])

print("=== Schema (name column excluded — direct identifier) ===")
print(df_pts.dtypes)
print(f"\nRow count: {len(df_pts):,}")

=== Schema (name column excluded — direct identifier) ===
patient_id                                 str
primary_provider_id                        str
postcode                                   str
updated_at             datetime64[us, Etc/UTC]
dtype: object

Row count: 68


In [8]:
print("=== Null rates (%) ===")
rates = null_rates(_df_pts_full)
# Annotate PII columns so the intent is clear
rates.index = [
    f"{c} [MASKED — direct identifier]" if c == "name"
    else f"{c} [values masked — quasi-identifier]" if c == "postcode"
    else c
    for c in rates.index
]
print(rates.to_string())

=== Null rates (%) ===
patient_id                                     0.0
name [MASKED — direct identifier]              0.0
primary_provider_id                            0.0
postcode [values masked — quasi-identifier]    0.0
updated_at                                     0.0


In [9]:
n_rows = len(df_pts)
n_unique_patients = df_pts["patient_id"].nunique()
rows_per_patient = df_pts.groupby("patient_id").size()

print("=== SCD history (multiple rows per patient_id) ===")
print(f"Total rows:                  {n_rows:,}")
print(f"Unique patient_id:           {n_unique_patients:,}")
print(f"Patients with >1 row:        {(rows_per_patient > 1).sum():,}")
print(f"Max rows per patient_id:     {rows_per_patient.max()}")
print(f"Avg rows per patient_id:     {rows_per_patient.mean():.2f}")

=== SCD history (multiple rows per patient_id) ===
Total rows:                  68
Unique patient_id:           50
Patients with >1 row:        18
Max rows per patient_id:     2
Avg rows per patient_id:     1.36


In [ ]:
# Verify that (patient_id, updated_at) is a unique composite key.
#
# Each SCD row should represent a distinct point-in-time snapshot.
# If two rows share (patient_id, updated_at) they are true duplicates —
# the incremental watermark cannot distinguish them, and dlt append would double-load them.

n_composite_unique = df_pts.drop_duplicates(subset=["patient_id", "updated_at"]).shape[0]
n_composite_dupes = n_rows - n_composite_unique

print("=== Composite key uniqueness: (patient_id, updated_at) ===")
print(f"Total rows:                      {n_rows}")
print(f"Unique (patient_id, updated_at): {n_composite_unique}")
print(f"Duplicate composite key rows:    {n_composite_dupes}")
print()

if n_composite_dupes == 0:
    print("✓ (patient_id, updated_at) is unique — every SCD row represents a distinct snapshot.")
    print("  dlt append + incremental cursor on updated_at is safe: no phantom duplicates.")
else:
    print("⚠️  Duplicate (patient_id, updated_at) pairs found — watermark-based dedup is unreliable.")
    dup_mask = df_pts.duplicated(subset=["patient_id", "updated_at"], keep=False)
    print(df_pts[dup_mask][["patient_id", "updated_at"]].sort_values(["patient_id", "updated_at"]).to_string(index=False))

In [10]:
print("=== Watermark coverage (updated_at) ===")
ts = df_pts["updated_at"]
print(f"min: {ts.min()}")
print(f"max: {ts.max()}")
print(f"null count: {ts.isnull().sum()}")

=== Watermark coverage (updated_at) ===
min: 2025-05-13 06:11:21.167810+00:00
max: 2026-05-06 06:11:21.167810+00:00
null count: 0


---
## 3. `providers`

**Key question:** Is `provider_id` truly unique? What specialties exist and how are they distributed?

In [11]:
df_prov = query("SELECT * FROM providers")

print("=== Schema ===")
print(df_prov.dtypes)

n_prov = len(df_prov)
n_unique_prov = df_prov["provider_id"].nunique()
print(f"\nRow count:           {n_prov:,}")
print(f"Unique provider_id:  {n_unique_prov:,}")
print(f"PK unique:           {n_prov == n_unique_prov}")

=== Schema ===
provider_id    str
name           str
specialty      str
dtype: object

Row count:           6
Unique provider_id:  6
PK unique:           True


In [12]:
print("=== Null rates (%) ===")
print(null_rates(df_prov).to_string())

print("\n=== Specialty distribution ===")
print(df_prov["specialty"].value_counts().to_string())

=== Null rates (%) ===
provider_id    0.0
name           0.0
specialty      0.0

=== Specialty distribution ===
specialty
GP             2
Cardiology     1
Pediatrics     1
Dermatology    1
Oncology       1


---
## 4. `patient_consents`

**Key question:** Is `patient_id` a true PK here? How do the three consent flags distribute across patients?

In [13]:
df_con = query("SELECT * FROM patient_consents")

print("=== Schema ===")
print(df_con.dtypes)

n_con = len(df_con)
n_unique_con = df_con["patient_id"].nunique()
print(f"\nRow count:          {n_con:,}")
print(f"Unique patient_id:  {n_unique_con:,}")
print(f"PK unique:          {n_con == n_unique_con}")

=== Schema ===
patient_id                str
consent_research         bool
consent_marketing        bool
consent_partner_share    bool
dtype: object

Row count:          50
Unique patient_id:  50
PK unique:          True


In [14]:
print("=== Null rates (%) ===")
print(null_rates(df_con).to_string())

print("\n=== Consent flag distributions ===")
for flag in ["consent_research", "consent_marketing", "consent_partner_share"]:
    total_flag = df_con[flag].notna().sum()
    true_count = df_con[flag].sum()
    false_count = total_flag - true_count
    pct = true_count / total_flag * 100 if total_flag else 0
    print(f"  {flag}: True={int(true_count)} ({pct:.1f}%)  False={int(false_count)}  null={df_con[flag].isnull().sum()}")

=== Null rates (%) ===
patient_id               0.0
consent_research         0.0
consent_marketing        0.0
consent_partner_share    0.0

=== Consent flag distributions ===
  consent_research: True=31 (62.0%)  False=19  null=0
  consent_marketing: True=15 (30.0%)  False=35  null=0
  consent_partner_share: True=25 (50.0%)  False=25  null=0


---
## 5. Cross-table integrity

Check FK coverage and consent table coverage across tables.

In [15]:
all_patient_ids = set(query("SELECT DISTINCT patient_id FROM patients")["patient_id"])
apt_patient_ids = set(df_apt["patient_id"].unique())
consent_patient_ids = set(df_con["patient_id"].unique())
prov_ids = set(df_prov["provider_id"].unique())
apt_provider_ids = set(df_apt["provider_id"].unique())

print("=== FK coverage ===")
orphan_apt_patients = apt_patient_ids - all_patient_ids
print(f"appointments.patient_id not in patients:    {len(orphan_apt_patients)}")

orphan_apt_providers = apt_provider_ids - prov_ids
print(f"appointments.provider_id not in providers:  {len(orphan_apt_providers)}")

orphan_consent = consent_patient_ids - all_patient_ids
print(f"patient_consents.patient_id not in patients: {len(orphan_consent)}")

print("\n=== Consent coverage ===")
patients_without_consent = all_patient_ids - consent_patient_ids
print(f"Unique patients without a consent record: {len(patients_without_consent)}")
print(f"Consent coverage: {len(consent_patient_ids)}/{len(all_patient_ids)} patients ({len(consent_patient_ids)/len(all_patient_ids)*100:.1f}%)")

=== FK coverage ===
appointments.patient_id not in patients:    0
appointments.provider_id not in providers:  0
patient_consents.patient_id not in patients: 0

=== Consent coverage ===
Unique patients without a consent record: 0
Consent coverage: 50/50 patients (100.0%)


In [16]:
conn.close()
print("Connection closed.")

Connection closed.


---
## Findings Summary

> **Instructions:** Run all cells above, then complete this cell with actual values from the output. This cell gates Phase 3 and Phase 4 design — do not proceed without it.

### Data quality issues

| Table | Finding | Impact |
|---|---|---|
| `appointments` | `event_id` duplicates: **X rows (Y%)** | Confirms business-dedup in `intm` when serve requires it |
| `appointments` | Status values observed: **...** | Confirm `cancelled` is a status, not a delete |
| `patients` | Patients with >1 row: **X** | Confirms source SCD2 feed — `append` write_disposition correct |
| `patients` | `updated_at` null count: **X** | Watermark reliability |
| `providers` | Null fields: **...** | Schema contract risk |
| `patient_consents` | PK unique: **True/False** | If False → SCD2 strategy needs revisit |

### Consent distribution

| Flag | True % | Notes |
|---|---|---|
| `consent_research` | **X%** | |
| `consent_marketing` | **X%** | |
| `consent_partner_share` | **X%** | |

### FK / referential integrity

- Orphan appointment patient_ids: **X**
- Orphan appointment provider_ids: **X**
- Patients without consent record: **X** — analytical outputs gated on consent must treat missing as non-consented

### Decisions / risks flagged for `tech_constraints.md`

- [ ] *(Record any surprises that change the Phase 3/4 design here)*
- [ ] Confirm whether patients without a consent record should be treated as non-consented by default (impacts serve model filter logic)